# design generator

this notebook builds the genetic algorithm that generates and evolves rocket designs.

the GA sits at the top of the pipeline:

```
generate candidate        ← this notebook
    → validate_rocket     (structural — src/structure.py)
    → filter_rocket       (analytic — src/filters.py)
    → [KSP simulation]    (ground truth — later)
```

---

## what we're building

six functions, built in order:

| function | purpose |
|----------|---------|
| `generate_random_rocket` | produce a random rocket dict with no validation — GA learns from zero |
| `score_rocket` | fitness function — returns `compute_delta_v` if rocket passes all checks, else 0 |
| `tournament_select` | pick survivors from a population by tournament |
| `mutate` | apply one random valid mutation to a rocket |
| `crossover` | stage-level crossover between two parent rockets |
| `run_ga` | the main loop — initialize, evaluate, select, reproduce, repeat |

---

## key design decisions

**fitness = delta-v (or 0).** rockets that fail the structural validator or analytic filters score 0. no partial credit for near-misses. the GA starts from completely random designs and learns validity through selection pressure.

**selection = tournament.** pick M random candidates, keep the best one, repeat. preserves population diversity better than fitness-proportionate (roulette wheel) selection, and doesn't require normalized fitness scores.

**crossover = stage-level.** take upper stage(s) from parent A and lower stage(s) from parent B. invalid offspring are discarded — validation is instantaneous so this costs nothing.

---

## setup

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '...')

from src.config import load_parts_by_name, load_resource_lookup

parts_by_name = load_parts_by_name()
resource_lookup = load_resource_lookup()



In [ ]:
print(parts_by_name['mk1-3pod']['nodes'])
print(parts_by_name['fuelTank']['nodes'])
print(parts_by_name['liquidEngine']['nodes'])

generate random rocket

In [ ]:
pods = []
tanks = []
engines = []
decouplers = []
for part in parts_by_name:
    if parts_by_name[part]['category'] == 'Pods':
        pods.append(part)
    if parts_by_name[part]['resources'] is not None and parts_by_name[part]['engine'] is None:
        tanks.append(part)
    if parts_by_name[part]['engine'] is not None:
        engines.append(part)
    if part.startswith('Decoupler_'):
        decouplers.append(part)


print(len(pods))
print(len(tanks))
print(len(engines))
print(len(decouplers))

In [ ]:
import random
from src.rocket import Rocket

def generate_random_rocket(parts_by_name: dict,
                           pods: list,
                           tanks: list,
                           engines: list,
                           decouplers: list,
                           max_stages:int = 2):

    r = Rocket(parts_by_name)

    pod = random.choice(pods)
    r.add_part('pod_0', pod, parent = None)

    tank_count = engine_count = decoupler_count = 0

    n_stages = random.randint(1, max_stages)
    current_parent = 'pod_0'
    for stage in range(n_stages):
        stage_num = stage
        tank = random.choice(tanks)
        r.add_part(f'tank_{tank_count}', tank, parent = current_parent, attach_node='bottom')
        current_parent = f'tank_{tank_count}'
        tank_count += 1

        engine = random.choice(engines)
        r.add_part(f'eng_{engine_count}', engine, parent = current_parent, attach_node='bottom')
        r.set_stage(f'eng_{engine_count}', stage_num)
        current_parent = f'eng_{engine_count}'
        engine_count += 1
        if stage != n_stages - 1:
            decup = random.choice(decouplers)
            r.add_part(f'decoupler_{decoupler_count}', decup, parent = current_parent, attach_node='bottom')
            r.set_stage(f'decoupler_{decoupler_count}', stage + 1)
            current_parent = f'decoupler_{decoupler_count}'
            decoupler_count += 1

    return r.to_dict()


generate_random_rocket(parts_by_name, pods, tanks, engines, decouplers)

score rocket

In [ ]:
from src.structure import validate_rocket
from src.filters import filter_rocket, DV_THRESHOLDS, compute_delta_v

DV_THRESHOLDS = DV_THRESHOLDS


def score_rocket(rocket_dict: dict,
                 parts_by_name: dict,
                 resource_lookup: dict):

    parts_list = [p['type'] for p in rocket_dict['parts']]
    is_valid = validate_rocket(rocket_dict, parts_by_name)
    is_filtered, errors = filter_rocket(rocket_dict, parts_list, parts_by_name, resource_lookup, DV_THRESHOLDS)
    if not is_valid or not is_filtered:
        return 0
    score = compute_delta_v(rocket_dict, parts_list, parts_by_name, resource_lookup)
    return score

rocket = generate_random_rocket(parts_by_name, pods, tanks, engines, decouplers)
score = score_rocket(rocket, parts_by_name, resource_lookup)
print(rocket)
print(score)


In [ ]:
from src.filters import compute_delta_v
toy_rocket = {
    "parts": [
        {"id": "pod_0",        "type": "mk1-3pod",    "parent": None},
        {"id": "tank_upper",   "type": "fuelTank",    "parent": "pod_0",      "attach_node": "bottom"},
        {"id": "eng_upper",    "type": "liquidEngine", "parent": "tank_upper", "attach_node": "bottom"},
        {"id": "decoupler_0",  "type": "Decoupler_1", "parent": "eng_upper",  "attach_node": "bottom"},
        {"id": "tank_booster", "type": "fuelTank",    "parent": "decoupler_0","attach_node": "bottom"},
        {"id": "eng_booster",  "type": "liquidEngine", "parent": "tank_booster","attach_node": "bottom"},
    ],
    "stages": {
        "eng_booster":  1,   # fires first
        "decoupler_0":  1,   # jettisons with booster stage
        "eng_upper":    0,   # fires last
    }
}

parts_list = [p['type'] for p in toy_rocket['parts']]
print(compute_delta_v(toy_rocket, parts_list, parts_by_name, resource_lookup))
print(compute_delta_v(toy_rocket, parts_list, parts_by_name, resource_lookup, return_breakdown=True))

  def evaluate_population(n, parts_by_name, resource_lookup,
                          pods, tanks, engines, decouplers,
                          max_stages=2, generation=0, detailed=False):

  Logic:
  1. loop n times, call generate_random_rocket each iteration
  2. run validate_rocket and filter_rocket separately (so we can record each)
  3. if valid and filtered: score = compute_delta_v(...), else score = 0
  4. build meta — always {'score': score}, add the extra fields if detailed=True
  5. append (rocket_dict, meta) to the list
  6. return the list


In [ ]:
def evaluate_population(n_rockets: int,
                         parts_by_name: dict,
                         resource_lookup: dict,
                         pods: list,
                         tanks: list,
                         engines: list,
                         decouplers: list,
                         max_stages: int = 2,
                         generation: int = 0,
                         detailed: bool = False):
    population = []
    for r in range(n_rockets):
        rocket = generate_random_rocket(parts_by_name, pods, tanks, engines, decouplers, max_stages=max_stages)
        valid = validate_rocket(rocket, parts_by_name)
        parts_list = [p['type'] for p in rocket['parts']]
        filtered, reasons = filter_rocket(rocket, parts_list, parts_by_name, resource_lookup, DV_THRESHOLDS)

        if valid and filtered:
            stage_dvs = compute_delta_v(rocket, parts_list, parts_by_name, resource_lookup, return_breakdown=True)
            score = sum(stage_dvs.values())
        else:
            score = 0
            stage_dvs = {}

        if not detailed:
            meta = {'score': score}
        else:
            meta = {'score': score,
                    'valid': valid,
                    'filtered': filtered,
                    'n_stages': len(set(rocket['stages'].values())),
                    'n_parts': len(parts_list),
                    'stage_dv': stage_dvs,
                    'generation': generation
                    }
        population.append((rocket, meta))
    return population


In [ ]:
pop = evaluate_population(10, parts_by_name, resource_lookup, pods, tanks, engines, decouplers, detailed=True)

for rocket, meta in pop:
    print(meta)
    print(sum(meta['stage_dv'].values()))


In [ ]:
def tournament_select(population: dict,
                      pct_survivors: float = 0.5,
                      tournament_size: int = 3):

    n_survivors = int(len(population) * pct_survivors)
    survivors = []
    while len(survivors) < n_survivors:
        competitors = random.choices(population, k = tournament_size)
        winner = max(competitors, key = lambda x: x[1]['score'])
        survivors.append(winner)
    return survivors


In [ ]:
pop = evaluate_population(1000, parts_by_name, resource_lookup, pods, tanks, engines, decouplers, detailed=True)
survivors = tournament_select(pop)
print(len(survivors))
print([s[1]['score'] for s in survivors])

mutate survivors

In [ ]:
import copy
def mutate_swap_part(rocket_dict: dict,
                     pods: list,
                     tanks: list,
                     engines: list,
                     decouplers: list):

    new_rocket = copy.deepcopy(rocket_dict)
    parts_list = [p for p in rocket_dict['parts'] if p['id'] != 'pod_0']
    rand_part = random.choice(parts_list)
    rand_part_type = rand_part['type']
    rand_part_id = rand_part['id']
    if rand_part_type in pods:
        swap = random.choice(pods)
    if rand_part_type in engines:
        swap = random.choice(engines)
    if rand_part_type in tanks:
        swap = random.choice(tanks)
    if rand_part_type in decouplers:
        swap = random.choice(decouplers)

    for i, p in enumerate(new_rocket['parts']):
        if p['id'] == rand_part_id:
            new_rocket['parts'][i]['type'] = swap
            break
    return new_rocket

for _ in range(10):
    rocket, meta = random.choice(pop)
    mutated = mutate_swap_part(rocket, pods, tanks, engines, decouplers)
    assert rocket['parts'][0]['type'] == mutated['parts'][0]['type'], "pod was swapped!"
    print("all good")

In [ ]:
def mutate_add_stage(rocket_dict: dict,
                     tanks: list,
                     engines: list,
                     decouplers: list,
                     max_stages = 4):

    new_rocket = copy.deepcopy(rocket_dict)
    n_stages = max(new_rocket['stages'].values())
    if len(set(new_rocket['stages'].values()))  == max_stages:
        return new_rocket

    #### find bottom part to make parent of new stage
    all_ids = {p['id'] for p in new_rocket['parts']}
    parent_ids = {p['parent'] for p in new_rocket['parts'] if p['parent'] is not None}
    bottom_id = (all_ids - parent_ids).pop()

    new_tank = random.choice(tanks)
    new_engine = random.choice(engines)
    new_decoupler = random.choice(decouplers)

    next_stage = n_stages + 1

    n_decouplers = sum(1 for p in new_rocket['parts'] if p['id'].startswith('decoupler_'))
    n_tanks = sum(1 for p in new_rocket['parts'] if p['id'].startswith('tank_'))
    n_engines = sum(1 for p in new_rocket['parts'] if p['id'].startswith('eng_'))

    new_rocket['parts'].append({'id': f"decoupler_{n_decouplers}",'type': new_decoupler, 'parent': bottom_id, 'attach_node': 'bottom'})
    new_rocket['parts'].append({'id': f"tank_{n_tanks}",'type': new_tank, 'parent': f"decoupler_{n_decouplers}", 'attach_node': 'bottom'})
    new_rocket['parts'].append({'id': f"eng_{n_engines}",'type': new_engine, 'parent':f"tank_{n_tanks}", 'attach_node': 'bottom'})

    new_rocket['stages'][f"decoupler_{n_decouplers}"] =  next_stage
    new_rocket['stages'][f"eng_{n_engines}"] = next_stage

    return new_rocket

rocket, meta = random.choice(pop)
print('before:', len(rocket['parts']), 'parts,', rocket['stages'])
mutated = mutate_add_stage(rocket, tanks, engines, decouplers)
print('after: ', len(mutated['parts']), 'parts,', mutated['stages'])

In [ ]:
def mutate_remove_stage(rocket_dict: dict):

    new_rocket = copy.deepcopy(rocket_dict)
    id_to_parent = {p['id']: p['parent'] for p in new_rocket['parts']}
    n_stages = max(new_rocket['stages'].values())
    if n_stages == 0:
        return new_rocket

    all_ids = {p['id'] for p in new_rocket['parts']}
    parent_ids = {p['parent'] for p in new_rocket['parts'] if p['parent'] is not None}
    bottom_id = (all_ids - parent_ids).pop()

    bottom_tank = id_to_parent[bottom_id]
    bottom_decoupler = id_to_parent[bottom_tank]
    to_remove = {bottom_id, bottom_tank, bottom_decoupler}

    new_rocket['parts'] = [p for p in new_rocket['parts'] if p['id'] not in to_remove]
    for pid in to_remove:
        new_rocket['stages'].pop(pid, None)

    return new_rocket

rocket, meta = random.choice(pop)
print('before:', len(rocket['parts']), 'parts,', rocket['stages'])
mutated = mutate_remove_stage(rocket)
print('after: ', len(mutated['parts']), 'parts,', mutated['stages'])


In [ ]:
def crossover(parent_a: tuple,
              parent_b: tuple):

    parent_a_dict, _ = parent_a
    parent_b_dict, _ = parent_b

    parent_a_copy = copy.deepcopy(parent_a_dict)
    parent_b_copy = copy.deepcopy(parent_b_dict)

    max_stage_a = max(parent_a_copy['stages'].values())
    a_cut = random.randint(0, max_stage_a)

    if max(parent_b_copy['stages'].values()) == 0:
        return (parent_a_copy, {'score': 0})

    child = copy.deepcopy(parent_a_copy)
    while max(child['stages'].values()) > a_cut:
        child = mutate_remove_stage(child)

    # find child's bottom after cut
    all_ids_child = {p['id'] for p in child['parts']}
    parent_ids_child = {p['parent'] for p in child['parts'] if p['parent'] is not None}
    bottom_id_child = (all_ids_child - parent_ids_child).pop()

    # collect graft parts from B (everything above stage 0)
    id_to_parent_b = {p['id']: p['parent'] for p in parent_b_copy['parts']}
    b_parts_by_id = {p['id']: p for p in parent_b_copy['parts']}

    all_ids_b = {p['id'] for p in parent_b_copy['parts']}
    parent_ids_b = {p['parent'] for p in parent_b_copy['parts'] if p['parent'] is not None}
    bottom_id_b = (all_ids_b - parent_ids_b).pop()

    stage_0_ids = {pid for pid, s in parent_b_copy['stages'].items() if s == 0}

    current = bottom_id_b
    graft_ids = []
    while True:
        graft_ids.append(current)
        parent = id_to_parent_b[current]
        if parent in stage_0_ids:
            break
        current = parent

    graft_ids = graft_ids[::-1]  # reverse to top-to-bottom order (decoupler first)

    # build old_id -> new_id mapping using child part counts
    n_decouplers = sum(1 for p in child['parts'] if p['id'].startswith('decoupler_'))
    n_tanks = sum(1 for p in child['parts'] if p['id'].startswith('tank_'))
    n_engines = sum(1 for p in child['parts'] if p['id'].startswith('eng_'))

    id_map = {}
    for old_id in graft_ids:
        if old_id.startswith('decoupler_'):
            id_map[old_id] = f"decoupler_{n_decouplers}"
            n_decouplers += 1
        elif old_id.startswith('tank_'):
            id_map[old_id] = f"tank_{n_tanks}"
            n_tanks += 1
        elif old_id.startswith('eng_'):
            id_map[old_id] = f"eng_{n_engines}"
            n_engines += 1

    # append grafted parts to child, re-parenting the topmost to child's bottom
    for i, old_id in enumerate(graft_ids):
        part = copy.deepcopy(b_parts_by_id[old_id])
        part['id'] = id_map[old_id]
        if i == 0:
            part['parent'] = bottom_id_child
        else:
            part['parent'] = id_map[graft_ids[i - 1]]
        child['parts'].append(part)

    # update stages dict, offsetting B's stage numbers by a_cut
    for old_id in graft_ids:
        if old_id in parent_b_copy['stages']:
            old_stage = parent_b_copy['stages'][old_id]
            child['stages'][id_map[old_id]] = old_stage + a_cut

    return (child, {'score': 0})

In [ ]:
parent_a = random.choice(survivors)
parent_b = random.choice(survivors)
child, meta = crossover(parent_a, parent_b)
print('A stages:', parent_a[0]['stages'])
print('B stages:', parent_b[0]['stages'])
print('child parts:', len(child['parts']), 'stages:', child['stages'])


In [ ]:
def mutate(rocket_dict: dict,
           pods: list,
           tanks: list,
           engines: list,
           decouplers: list,
           max_stages: int = 4):

    choices = ['swap','add','remove']
    mutation = random.choice(choices)

    if mutation == 'swap':
        return mutate_swap_part(rocket_dict, pods, tanks, engines, decouplers)
    if mutation == 'add':
        return mutate_add_stage(rocket_dict, tanks, engines, decouplers, max_stages = max_stages)
    if mutation == 'remove':
        return mutate_remove_stage(rocket_dict)



In [ ]:
def score_population(rockets: list,
                     parts_by_name: dict,
                     resource_lookup: dict,
                     generation: int = 0,
                     detailed: bool =False):

    population = []
    for rocket in rockets:
        valid = validate_rocket(rocket, parts_by_name)
        parts_list = [p['type'] for p in rocket['parts']]
        filtered, reasons = filter_rocket(rocket, parts_list, parts_by_name, resource_lookup, DV_THRESHOLDS)

        if valid and filtered:
            stage_dvs = compute_delta_v(rocket, parts_list, parts_by_name, resource_lookup, return_breakdown=True)
            score = sum(stage_dvs.values())
        else:
            score = 0
            stage_dvs = {}

        if not detailed:
            meta = {'score': score}
        else:
            meta = {'score': score,
                    'valid': valid,
                    'filtered': filtered,
                    'n_stages': len(set(rocket['stages'].values())),
                    'n_parts': len(parts_list),
                    'stage_dv': stage_dvs,
                    'generation': generation
                    }
        population.append((rocket, meta))
    return population


In [ ]:
def run_ga(n_rockets: int,
           n_generations: int,
           parts_by_name: dict,
           resource_lookup: dict,
           pods: list,
           tanks: list,
           engines: list,
           decouplers: list,
           max_stages: int = 2,
           n_elites: int = 5,
           mutation_rate: float = 0.3,
           detailed: bool = False,
           save_dir: str = None):

    population = evaluate_population(n_rockets, parts_by_name, resource_lookup, pods, tanks, engines, decouplers, max_stages=max_stages, generation=0, detailed=detailed)

    if save_dir:
        save_generation(population, 0, save_dir)

    for gen in range(n_generations):
        children = []
        survivors = tournament_select(population)
        elites = sorted(survivors, key = lambda x: x[1]['score'], reverse = True)[:n_elites]
        while len(children) < n_rockets - n_elites:
            parent_a = random.choice(survivors)
            parent_b = random.choice(survivors)
            child, _ = crossover(parent_a, parent_b)
            if random.random() < mutation_rate:
                child = mutate(child, pods=pods, tanks=tanks, engines=engines, decouplers=decouplers, max_stages=max_stages)
            children.append(child)

        population = score_population(rockets=children, parts_by_name=parts_by_name, resource_lookup=resource_lookup, detailed=detailed, generation=gen+1)
        population.extend(elites)

        if save_dir:
            save_generation(population, gen+1, save_dir)

    return population

In [ ]:
result = run_ga(100, 5, parts_by_name, resource_lookup, pods, tanks, engines, decouplers, detailed=True)
scores = [meta['score'] for _, meta in result]
print(f"final gen — best: {max(scores):.0f}, mean: {sum(scores)/len(scores):.0f}, nonzero: {sum(1 for s in scores if s > 0)}")

In [ ]:
import json
import os

def save_generation(population: list,
                    generation: int,
                    run_dir: str):

    os.makedirs(run_dir, exist_ok=True)
    records = [{'rocket': rocket, 'meta': meta} for rocket, meta in population]
    filename = os.path.join(run_dir, f"gen_{generation:03d}.json")
    with open(filename, 'w') as f:
        json.dump({'generation': generation, 'rockets': records}, f, indent=2)
    print(f"saved {len(population)} rockets to {filename}")

In [ ]:
import matplotlib.pyplot as plt
import glob
import numpy as np

def plot_run(run_dir: str):
    gen_files = sorted(glob.glob(os.path.join(run_dir, 'gen_*.json')))
    if not gen_files:
        print(f"no generation files found in {run_dir}")
        return

    generations = []
    all_scores = []
    means = []
    n_zeros = []
    n_invalid = []

    for f in gen_files:
        with open(f) as fh:
            data = json.load(fh)
        gen = data['generation']
        scores = [r['meta']['score'] for r in data['rockets']]
        invalids = [r for r in data['rockets'] if not r['meta'].get('valid', True)]

        generations.append(gen)
        all_scores.append(scores)
        means.append(sum(scores) / len(scores))
        n_zeros.append(sum(1 for s in scores if s == 0))
        n_invalid.append(len(invalids))

    # clip y-axis at 99th percentile, mark outliers with triangles at the top
    flat_scores = [s for gen_scores in all_scores for s in gen_scores if s > 0]
    y_cap = np.percentile(flat_scores, 99) * 1.1 if flat_scores else 1

    fig, ax = plt.subplots(figsize=(12, 6))

    for gen, scores in zip(generations, all_scores):
        clipped = [min(s, y_cap) for s in scores]
        outlier_mask = [s > y_cap for s in scores]
        ax.scatter([gen] * len(clipped), clipped, alpha=0.2, s=10, color='steelblue')
        if any(outlier_mask):
            ax.scatter([gen] * sum(outlier_mask), [y_cap] * sum(outlier_mask),
                       marker='^', color='orange', s=30, zorder=5, label='outlier (clipped)' if gen == generations[0] else '')

    ax.plot(generations, means, color='tomato', linewidth=2, label='mean delta-v')
    ax.set_ylim(0, y_cap * 1.15)

    for gen, mean, nz, ni in zip(generations, means, n_zeros, n_invalid):
        ax.text(gen, y_cap * 1.08, f"mean:{mean:.0f}\nzeros:{nz}\ninvalid:{ni}",
                ha='center', va='top', fontsize=7, color='dimgray')

    ax.set_xlabel('generation')
    ax.set_ylabel('delta-v (m/s)')
    ax.set_title('GA run — delta-v by generation')
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys())
    plt.tight_layout()
    plt.show()

In [ ]:
from datetime import datetime
run_dir = f"../data/runs/run_{datetime.now().strftime('%Y-%m-%d-%H%M')}"
result = run_ga(100, 7, parts_by_name, resource_lookup, pods, tanks, engines, decouplers, detailed=True, save_dir=run_dir)
plot_run(run_dir)